In [1]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataset import prepare_data, TARGET_COL

sns.set_theme(style="whitegrid", palette="muted")

data = prepare_data()
print("Data loaded.")


Dataset loaded: 5875 rows, 19 features
Target range: 7.00 – 54.99
Subjects: 42
Split complete (subject-wise, no leakage):
  Train: 3931 rows | 28 subjects
  Val  : 709 rows | 5 subjects
  Test : 1235 rows | 9 subjects
Scaler saved to /Users/sohamwarad/Documents/GitHub/Parkinson's UPDRS Severity Estimator/notebooks/../models/scaler.pkl
Data loaded.


In [2]:
X_train = data['X_train']
X_val   = data['X_val']
X_test  = data['X_test']
y_train = data['y_train']
y_val   = data['y_val']
y_test  = data['y_test']
feature_cols = data['feature_cols']

print("=== Train set (should be ~mean=0, std=1) ===")
print(f"  Mean: {X_train.mean():.4f}  (should be ~0.0)")
print(f"  Std : {X_train.std():.4f}   (should be ~1.0)")

print("\n=== Val set (scaled using train scaler) ===")
print(f"  Mean: {X_val.mean():.4f}")
print(f"  Std : {X_val.std():.4f}")

print("\n=== Test set ===")
print(f"  Mean: {X_test.mean():.4f}")
print(f"  Std : {X_test.std():.4f}")

print("\n=== Target (y) — should NOT be scaled ===")
print(f"  y_train range: {y_train.min():.2f} – {y_train.max():.2f}")
print(f"  y_test  range: {y_test.min():.2f} – {y_test.max():.2f}")

=== Train set (should be ~mean=0, std=1) ===
  Mean: -0.0000  (should be ~0.0)
  Std : 1.0000   (should be ~1.0)

=== Val set (scaled using train scaler) ===
  Mean: 0.0516
  Std : 0.9119

=== Test set ===
  Mean: 0.1254
  Std : 1.0491

=== Target (y) — should NOT be scaled ===
  y_train range: 7.00 – 53.96
  y_test  range: 10.65 – 44.76


In [3]:
from dataset import load_raw, get_feature_cols

df = load_raw()
feature_cols = get_feature_cols(df)

print("=== Missing values ===")
missing = df[feature_cols].isnull().sum()
print(missing[missing > 0] if missing.any() else "  None — dataset is complete ✓")

print("\n=== Outlier check (features with values > 5 std from mean) ===")
from scipy import stats
z_scores = np.abs(stats.zscore(df[feature_cols]))
outlier_counts = (z_scores > 5).sum()
outliers = outlier_counts[outlier_counts > 0]
if len(outliers) > 0:
    print(outliers)
else:
    print("  No extreme outliers (>5σ) found ✓")

print("\n=== Class balance across severity bins ===")
def severity_bin(score):
    if score < 20: return 'Mild'
    elif score < 36: return 'Moderate'
    else: return 'Severe'

df['severity'] = df['total_UPDRS'].apply(severity_bin)
counts = df['severity'].value_counts()
print(counts)
print(f"\n  Mild     : {counts.get('Mild',0)/len(df)*100:.1f}%")
print(f"  Moderate : {counts.get('Moderate',0)/len(df)*100:.1f}%")
print(f"  Severe   : {counts.get('Severe',0)/len(df)*100:.1f}%")

=== Missing values ===
  None — dataset is complete ✓

=== Outlier check (features with values > 5 std from mean) ===
[575]

=== Class balance across severity bins ===
severity
Moderate    3103
Severe      1496
Mild        1276
Name: count, dtype: int64

  Mild     : 21.7%
  Moderate : 52.8%
  Severe   : 25.5%


In [4]:
print("=== Which features have >5σ outliers ===")
outlier_per_feature = (z_scores > 5).sum(axis=0)
outlier_df = pd.Series(outlier_per_feature, index=feature_cols)
outlier_df = outlier_df[outlier_df > 0].sort_values(ascending=False)
print(outlier_df)

=== Which features have >5σ outliers ===
NHR              67
Shimmer:APQ5     58
Jitter:PPQ5      55
Shimmer          55
Shimmer(dB)      53
Jitter(%)        49
Shimmer:APQ3     42
Shimmer:DDA      42
Jitter:RAP       40
Jitter:DDP       40
Shimmer:APQ11    40
Jitter(Abs)      30
PPE               4
dtype: int64


In [5]:
from dataset import load_raw, subject_wise_split, SUBJECT_COL

df = load_raw()
train_df, val_df, test_df = subject_wise_split(df)

train_subj = set(train_df[SUBJECT_COL])
val_subj   = set(val_df[SUBJECT_COL])
test_subj  = set(test_df[SUBJECT_COL])

print("=== Subject overlap check ===")
print(f"  Train ∩ Val  : {train_subj & val_subj}  (should be empty)")
print(f"  Train ∩ Test : {train_subj & test_subj}  (should be empty)")
print(f"  Val   ∩ Test : {val_subj & test_subj}  (should be empty)")

print(f"\n=== Subject IDs ===")
print(f"  Train subjects : {sorted(train_subj)}")
print(f"  Val subjects   : {sorted(val_subj)}")
print(f"  Test subjects  : {sorted(test_subj)}")

Split complete (subject-wise, no leakage):
  Train: 3931 rows | 28 subjects
  Val  : 709 rows | 5 subjects
  Test : 1235 rows | 9 subjects
=== Subject overlap check ===
  Train ∩ Val  : set()  (should be empty)
  Train ∩ Test : set()  (should be empty)
  Val   ∩ Test : set()  (should be empty)

=== Subject IDs ===
  Train subjects : [1, 2, 3, 4, 6, 7, 8, 10, 12, 13, 15, 16, 17, 18, 21, 23, 24, 25, 28, 29, 32, 33, 34, 36, 37, 38, 39, 42]
  Val subjects   : [11, 19, 22, 35, 41]
  Test subjects  : [5, 9, 14, 20, 26, 27, 30, 31, 40]
